In [5]:
import pywt
import numpy as np
from scipy.signal import find_peaks

In [6]:
data_path = "../UCI/data"
FS = 125
WINDOW_SIZE = 1000

In [7]:
def extract_bp_from_abp(abp_window):

    # Detect systolic peaks
    peaks, _ = find_peaks(
        abp_window,
        distance=50,       # ~0.4 sec at 125 Hz
        prominence=5
    )


    # Need enough beats in the window
    if len(peaks) < 2:
        return None, None


    # SBP = pressure at systolic peaks
    sbp_values = abp_window[peaks]


    dbp_values = []


    # Find minimum pressure between consecutive systolic peaks
    for i in range(len(peaks)-1):

        beat_segment = abp_window[
            peaks[i]:peaks[i+1]
        ]

        if len(beat_segment) > 0:

            dbp_values.append(
                np.min(beat_segment)
            )


    if len(dbp_values) == 0:
        return None, None


    # Use median to reduce effect of noisy beats
    sbp = np.median(sbp_values)
    dbp = np.median(dbp_values)


    # Optional sanity check
    if sbp < 50 or sbp > 250:
        return None, None

    if dbp < 20 or dbp > 150:
        return None, None


    return sbp, dbp

In [8]:
def process_recording(
    recording,
    window_size=1000,
    step_size=500,
    fs=125
):

    data = recording[:]

    # -----------------------------------------
    # Extract channels
    # -----------------------------------------

    ppg = data[:, 0]
    abp = data[:, 1]

    # -----------------------------------------
    # Recording-level PPG normalization
    # -----------------------------------------

    mean = np.mean(ppg)
    std = np.std(ppg)

    if std == 0:
        return None, None, None

    ppg = (ppg - mean) / std

    X = []
    y = []
    delays = []

    # -----------------------------------------
    # Windowing
    # -----------------------------------------

    for start in range(
        0,
        len(ppg) - window_size + 1,
        step_size
    ):

        ppg_window = ppg[
            start:start + window_size
        ]

        abp_window = abp[
            start:start + window_size
        ]

        # -------------------------------------
        # Extract SBP / DBP
        # -------------------------------------

        sbp, dbp = extract_bp_from_abp(
            abp_window
        )

        if sbp is None:
            continue


        X.append(ppg_window)

        y.append([
            sbp,
            dbp
        ])

    if len(X) == 0:
        return None, None, None

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32))



In [9]:
def process_split(
    records,
    window_size=1000,
    step_size=500,
    fs=125
):
    """
    Process all recordings belonging to one split.

    Each recording is:
        1. Loaded
        2. PPG normalized at recording level
        3. Windowed
        4. SBP/DBP extracted from corresponding ABP window

    Returns:
        X : (N, 1000)
        y : (N, 2)
    """

    X_all = []
    y_all = []

    for idx, (f, reference) in enumerate(records):

        # -----------------------------------------
        # Load complete recording
        # -----------------------------------------

        recording = f[reference]

        # -----------------------------------------
        # Process recording
        # -----------------------------------------

        result = process_recording(
            recording,
            window_size=window_size,
            step_size=step_size,
            fs=fs
        )

        if result[0] is None:
            continue

        X_recording, y_recording = result

        # -----------------------------------------
        # Store windows from this recording
        # -----------------------------------------

        X_all.append(X_recording)
        y_all.append(y_recording)

        # -----------------------------------------
        # Progress
        # -----------------------------------------

        if (idx + 1) % 500 == 0:
            print(
                f"Processed {idx + 1}/{len(records)} recordings"
            )

    # -----------------------------------------
    # Combine all recordings
    # -----------------------------------------

    if len(X_all) == 0:
        return None, None

    X_all = np.concatenate(
        X_all,
        axis=0
    )

    y_all = np.concatenate(
        y_all,
        axis=0
    )

    return X_all, y_all

In [ ]:
def wavelet_denoise_db8(
    ppg,
    wavelet="db8",
    level=10
):

    ppg = np.asarray(
        ppg,
        dtype=np.float64
    )

    # -----------------------------------------
    # Check signal
    # -----------------------------------------

    if len(ppg) < 2:
        return None

    if not np.all(
        np.isfinite(ppg)
    ):
        return None

    # -----------------------------------------
    # DWT decomposition
    # -----------------------------------------

    coeffs = pywt.wavedec(
        ppg,
        wavelet=wavelet,
        level=level
    )

    # coeffs structure:
    #
    # [A10, D10, D9, ..., D2, D1]
    #
    # A10 = lowest-frequency approximation
    # D1  = highest-frequency detail

    # -----------------------------------------
    # Remove the lowest-frequency
    # approximation component
    #
    # This removes very slow baseline
    # variation.
    # -----------------------------------------

    coeffs[0] = np.zeros_like(
        coeffs[0]
    )

    # -----------------------------------------
    # Reconstruct signal
    # -----------------------------------------

    denoised = pywt.waverec(
        coeffs,
        wavelet=wavelet
    )

    # waverec can return slightly more samples
    # because of boundary handling.
    
    denoised = denoised[
        :len(ppg)
    ]

    return denoised.astype(
        np.float32
    )